<a href="https://colab.research.google.com/github/ClarenceBallensky/MLP-vs-GCN-ids/blob/main/Build_Sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Sample

## Imports

In [ ]:
import os, zipfile, hashlib
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive, files

## Build Sample (maintainer use only)
Rebuilds ids_sample.parquet from the full improved CSE-CIC-IDS2018 download.
Needs a lot of RAM and time. Not required to run the main notebook.

In [ ]:
drive.mount('/content/drive')

URL = "https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CSECICIDS2018_improved.zip"
FRAC = 0.10
TARGET_ROWS = 300_000
SEED = 42
OUT = "/content/drive/MyDrive/ids_samples"
os.makedirs(OUT, exist_ok=True)

## Step 1: download and sample 10% of each daily CSV (saved per file so a crash loses little)


In [ ]:
!wget -q -c -O /content/data.zip {URL}

with zipfile.ZipFile("/content/data.zip") as z:
    for name in z.namelist():
        out = f"{OUT}/{os.path.basename(name)}.parquet"
        if name.endswith(".csv") and not os.path.exists(out):
            chunks = pd.read_csv(z.open(name), chunksize=200_000)
            sample = [c.dropna().sample(frac=FRAC, random_state=SEED) for c in chunks]
            pd.concat(sample).to_parquet(out)
            print("done:", name)

!ls -lh {OUT}

## Step 2: combine into one ~300k-row file and print its hash

In [ ]:
parquet_files = sorted(f for f in os.listdir(OUT) if f.endswith(".parquet"))
total = sum(pq.ParquetFile(f"{OUT}/{f}").metadata.num_rows for f in parquet_files)
frac = min(1.0, TARGET_ROWS / total)
print(total, "rows available; keeping fraction", round(frac, 3))

df = pd.concat(
    pd.read_parquet(f"{OUT}/{f}").sample(frac=frac, random_state=SEED)
    for f in parquet_files
).reset_index(drop=True)

fl = df.select_dtypes("float64").columns
df[fl] = df[fl].astype("float32")

df.to_parquet("/content/ids_sample.parquet")
print(df.shape)
print(os.path.getsize("/content/ids_sample.parquet") / 1e6, "MB")
print("sha256:", hashlib.sha256(open("/content/ids_sample.parquet", "rb").read()).hexdigest())

files.download("/content/ids_sample.parquet")